In [1]:
import os
import json
import pandas as pd
import numpy as np
import xarray as xr
import s3fs
from pathlib import Path

ERA5_ZARR = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

AIRPORT = "JFK"
AIRPORT_LAT = 40.6413
AIRPORT_LON = -73.7781

START = "2022-01-01T00:00:00"
END = "2022-01-03T23:00:00"

VARIABLES = [
    "2m_temperature",
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "total_precipitation",
    "surface_pressure",
    "convective_available_potential_energy",
]

MINIO_ENDPOINT = "http://minio:9000"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"

In [2]:
def lon_to_360(lon):
    return lon % 360

airport_lon_360 = lon_to_360(AIRPORT_LON)
airport_lon_360

286.2219

In [3]:
ds = xr.open_zarr(
    ERA5_ZARR,
    chunks=None,
    storage_options={"token": "anon"},
)

print(ds)

<xarray.Dataset> Size: 4PB
Dimensions:                                                          (
                                                                      time: 1323648,
                                                                      latitude: 721,
                                                                      longitude: 1440,
                                                                      level: 37)
Coordinates:
  * time                                                             (time) datetime64[ns] 11MB ...
  * latitude                                                         (latitude) float32 3kB ...
  * longitude                                                        (longitude) float32 6kB ...
  * level                                                            (level) int64 296B ...
Data variables: (12/273)
    100m_u_component_of_wind                                         (time, latitude, longitude) float32 5TB ...
    100m_v_component_of_wind

In [4]:
missing = [v for v in VARIABLES if v not in ds.data_vars]
if missing:
    raise ValueError(f"Missing variables in ARCO-ERA5: {missing}")

print("All required variables exist")

All required variables exist


In [5]:
point = (
    ds[VARIABLES]
    .sel(latitude=AIRPORT_LAT, longitude=airport_lon_360, method="nearest")
    .sel(time=slice(START, END))
    .load()
)

pdf = point.to_dataframe().reset_index()

pdf["airport"] = AIRPORT
pdf["event_id"] = [
    f"{AIRPORT}_{str(ts).replace(' ', 'T')}" for ts in pdf["time"].astype(str)
]

pdf = pdf.rename(columns={
    "time": "timestamp_utc",
    "2m_temperature": "temperature_k",
    "10m_u_component_of_wind": "wind_u_ms",
    "10m_v_component_of_wind": "wind_v_ms",
    "total_precipitation": "precipitation_m",
    "surface_pressure": "surface_pressure_pa",
    "convective_available_potential_energy": "cape_j_kg",
})

pdf["temperature_c"] = pdf["temperature_k"] - 273.15
pdf["wind_speed_ms"] = np.sqrt(pdf["wind_u_ms"] ** 2 + pdf["wind_v_ms"] ** 2)
pdf["wind_speed_kts"] = pdf["wind_speed_ms"] * 1.94384
pdf["precipitation_mm"] = pdf["precipitation_m"] * 1000.0

pdf = pdf[
    [
        "event_id",
        "airport",
        "timestamp_utc",
        "latitude",
        "longitude",
        "temperature_k",
        "temperature_c",
        "wind_u_ms",
        "wind_v_ms",
        "wind_speed_ms",
        "wind_speed_kts",
        "precipitation_m",
        "precipitation_mm",
        "surface_pressure_pa",
        "cape_j_kg",
    ]
]

print(pdf.shape)
pdf.head()

(72, 15)


,event_id,airport,timestamp_utc,latitude,longitude,temperature_k,temperature_c,wind_u_ms,wind_v_ms,wind_speed_ms,wind_speed_kts,precipitation_m,precipitation_mm,surface_pressure_pa,cape_j_kg
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01 00:00:00,40.75,286.25,282.574158,9.424164,-0.005779,1.871856,1.871865,3.638606,5.761161e-06,0.005761,101271.765625,12.700439
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01 01:00:00,40.75,286.25,282.615082,9.465088,0.255980,2.030057,2.046132,3.977354,1.056492e-05,0.010565,101288.007812,21.743164
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01 02:00:00,40.75,286.25,282.552124,9.402130,-0.076426,2.341874,2.343121,4.554652,1.248531e-05,0.012485,101208.484375,23.470459
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01 03:00:00,40.75,286.25,282.542694,9.392700,0.014148,2.136289,2.136336,4.152695,1.439825e-06,0.001440,101199.937500,17.780762
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01 04:00:00,40.75,286.25,282.262512,9.112518,0.130988,2.419828,2.423371,4.710645,9.592623e-07,0.000959,101201.640625,14.834229


In [6]:
Path("data/sample").mkdir(parents=True, exist_ok=True)

sample_path = Path("data/sample/jfk_era5_2022_01_01_to_03.jsonl")

with sample_path.open("w") as f:
    for row in pdf.to_dict(orient="records"):
        row["timestamp_utc"] = str(row["timestamp_utc"])
        f.write(json.dumps(row) + "\n")

print("Wrote:", sample_path)

Wrote: data/sample/jfk_era5_2022_01_01_to_03.jsonl


In [7]:
fs = s3fs.S3FileSystem(
    key=MINIO_ACCESS_KEY,
    secret=MINIO_SECRET_KEY,
    client_kwargs={"endpoint_url": MINIO_ENDPOINT},
)

raw_parquet_path = "s3://raw/era5_sample/airport=JFK/year=2022/month=01/jfk_2022_01_01_to_03.parquet"

with fs.open(raw_parquet_path, "wb") as f:
    pdf.to_parquet(f, index=False)

print("Uploaded to MinIO:", raw_parquet_path)

Uploaded to MinIO: s3://raw/era5_sample/airport=JFK/year=2022/month=01/jfk_2022_01_01_to_03.parquet


In [8]:
with fs.open(raw_parquet_path, "rb") as f:
    check = pd.read_parquet(f)

print(check.shape)
check.head()

(72, 15)


,event_id,airport,timestamp_utc,latitude,longitude,temperature_k,temperature_c,wind_u_ms,wind_v_ms,wind_speed_ms,wind_speed_kts,precipitation_m,precipitation_mm,surface_pressure_pa,cape_j_kg
0,JFK_2022-01-01T00:00:00,JFK,2022-01-01 00:00:00,40.75,286.25,282.574158,9.424164,-0.005779,1.871856,1.871865,3.638606,5.761161e-06,0.005761,101271.765625,12.700439
1,JFK_2022-01-01T01:00:00,JFK,2022-01-01 01:00:00,40.75,286.25,282.615082,9.465088,0.255980,2.030057,2.046132,3.977354,1.056492e-05,0.010565,101288.007812,21.743164
2,JFK_2022-01-01T02:00:00,JFK,2022-01-01 02:00:00,40.75,286.25,282.552124,9.402130,-0.076426,2.341874,2.343121,4.554652,1.248531e-05,0.012485,101208.484375,23.470459
3,JFK_2022-01-01T03:00:00,JFK,2022-01-01 03:00:00,40.75,286.25,282.542694,9.392700,0.014148,2.136289,2.136336,4.152695,1.439825e-06,0.001440,101199.937500,17.780762
4,JFK_2022-01-01T04:00:00,JFK,2022-01-01 04:00:00,40.75,286.25,282.262512,9.112518,0.130988,2.419828,2.423371,4.710645,9.592623e-07,0.000959,101201.640625,14.834229
